### Load OMNIA Raw Energy Data

In [ ]:
import numpy as np
import pandas as pd
from iode import *
import iode as io
import re
import os

# Load and merge UNData into one dataframe
pathwork = "E:/Work/GitHub/NEMESIS-World/"
path_enr_data = pathwork + "data_raw/energy/"
file_enr = "Omnia_Energy_Balance_v20250212_trav.xlsx"
pathdata = path_enr_data+file_enr

Omnia_raw = pd.read_excel(pathdata, sheet_name = "ALL")
Omnia_trav = Omnia_raw.groupby(["Sector", "Sub-sector", "Enerp NeW", "Unit", "Year", "Region New"], as_index=False)["Value"].sum().copy()

### Process data to convert OMNIA data into NeW variables (DFINENP, DINTENP, etc.)

In [2]:
df_omnia = pd.DataFrame()

# Selection through sector names for DFINENP and DINTENP
# Concatenate name and energy product

# DFINENP
lst_subsec_dfinen = ["AGR","IND","RES","SRV", "TRA", "OTH", "Own uses"]
Omnia_dfinen = Omnia_trav.loc[Omnia_trav["Sector"].isin(lst_subsec_dfinen)].copy()
Omnia_dfinen["Variable_NeW"] = "DFINENP" + Omnia_dfinen["Enerp NeW"]
Omnia_dfinen.drop(columns=["Sector", "Enerp NeW"], inplace = True)
Omnia_dfinen.rename(columns={"Sub-sector":  "Sector", "Region New": "Region_NeW"}, inplace = True)
Omnia_dfinen = Omnia_dfinen[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]
#print(Omnia_dfinen.head())

# DINTEP
lst_subsec_dinten = ["TRF-IN"]
Omnia_dinten = Omnia_trav.loc[Omnia_trav["Sector"].isin(lst_subsec_dinten)].copy()
Omnia_dinten["Variable_NeW"] = "DINTENP" + Omnia_dinten["Enerp NeW"]
Omnia_dinten.drop(columns=["Sector", "Enerp NeW"], inplace = True)
Omnia_dinten.rename(columns={"Sub-sector":  "Sector", "Region New": "Region_NeW"}, inplace = True)
Omnia_dinten = Omnia_dinten[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]
#Omnia_dinten.head()


In [ ]:
# Specific data processing for DINTENP in PG & HEAT required (to split inputs thereafter)

# DINTEN PG & HEAT
lst_subsec_dinten_pg = ["PWR"]
lst_cmd_todrop_dinten_pg = ["ELEC", "HEAT"]
lst_var_chp = ["CHP AUTO", "CHP MAIN"]

#########
# Calculate the Electricitry and Heat production from CHP (to split inputs thereafter)
chp_calc = Omnia_trav.loc[(Omnia_trav["Sub-sector"].isin(lst_var_chp)) 
                          & ((Omnia_trav["Enerp NeW"].isin(lst_cmd_todrop_dinten_pg)))]
chp_calc = chp_calc.groupby(by=["Enerp NeW", "Unit", "Year", "Region New"], as_index= False)["Value"].sum()
for enrp in lst_cmd_todrop_dinten_pg:
    chp_calc.loc[chp_calc["Enerp NeW"] == f"{enrp}", "Variable_NeW"] = f"PRODP{enrp}"

# Pivot dataframe to calculate share of electricity (and heat) in CHP
chp_calc_pivoted = chp_calc.pivot_table(index=["Unit", "Year", "Region New"],columns="Enerp NeW", values="Value").reset_index()
chp_calc_pivoted["Share_ELEC"] = chp_calc_pivoted["ELEC"]/(chp_calc_pivoted["ELEC"]+chp_calc_pivoted["HEAT"])

# Calculate DINTENP for CHP 
chp_selec = Omnia_trav.loc[(Omnia_trav["Sub-sector"].isin(lst_var_chp)) 
                          & (~(Omnia_trav["Enerp NeW"].isin(lst_cmd_todrop_dinten_pg)))]
chp_selec = chp_selec.groupby(by=["Enerp NeW", "Unit", "Year", "Region New"], as_index= False)["Value"].sum()


# Merge dfs to calculate "DINTENP_PG" et "DINTENP_HT"
chp_merged = chp_selec.merge(chp_calc_pivoted, on=["Unit", "Year", "Region New"], how="left")
chp_merged["DINTENP_PG"] = chp_merged["Value"]*chp_merged["Share_ELEC"]
chp_merged["DINTENP_HT"] = chp_merged["Value"]*(1-chp_merged["Share_ELEC"])
chp_merged.drop(columns=["ELEC", "HEAT", "Value", "Share_ELEC"], inplace= True)


# Convert columuns "DINTENP_PG" & "DINTENP_HT" in rows
chp_dintenp = pd.melt(chp_merged, id_vars=["Enerp NeW", "Unit", "Year", "Region New"], value_vars=["DINTENP_PG", "DINTENP_HT"],
                var_name="Sector", value_name="Value")
chp_dintenp.loc[(chp_dintenp["Sector"] == "DINTENP_PG"), "Sector_agg"] = "PG"
chp_dintenp.loc[(chp_dintenp["Sector"] == "DINTENP_HT"), "Sector_agg"] = "HEAT"
chp_dintenp.drop(columns=["Sector"], inplace= True)
chp_dintenp = chp_dintenp[["Region New", "Sector_agg", "Enerp NeW", "Year", "Value", "Unit"]]


#########
# DINTENP (excep. CHP)
Omnia_dinten_selec = Omnia_trav.loc[(Omnia_trav["Sector"].isin(lst_subsec_dinten_pg)) 
                                 & (~(Omnia_trav["Enerp NeW"].isin(lst_cmd_todrop_dinten_pg))) 
                                 & (~(Omnia_trav["Sub-sector"].isin(lst_var_chp)))].copy()
dict_elec_ht = {'Electricity AUTO': "PG", 'Electricity MAIN': "PG", 'Heat AUTO': "HEAT", 'Heat MAIN': "HEAT"}
for var, cmd in dict_elec_ht.items():
    Omnia_dinten_selec.loc[Omnia_dinten_selec["Sub-sector"] == f"{var}", "Sector_agg"] = f"{cmd}"
Omnia_dinten_selec = Omnia_dinten_selec.groupby(by=["Region New", "Sector_agg", "Enerp NeW", "Year", "Unit"], as_index= False)["Value"].sum()


########
# Merge DINTENP CHP & Non-CHP¨
Omnia_dinten_tot = pd.concat([Omnia_dinten_selec, chp_dintenp])
Omnia_dinten_pg = Omnia_dinten_tot.groupby(by=["Region New", "Sector_agg", "Enerp NeW", "Year","Unit"], as_index = False)["Value"].sum()
Omnia_dinten_pg["Variable_NeW"] = "DINTENP" + Omnia_dinten_pg["Enerp NeW"]
Omnia_dinten_pg.drop(columns=["Enerp NeW"], inplace=True)
Omnia_dinten_pg.rename(columns={"Region New": "Region_NeW", "Sector_agg": "Sector"}, inplace= True)
Omnia_dinten_pg = Omnia_dinten_pg[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]

#########
# Merge DINTENP PG/HEAT with other DINTENP
Omnia_dinten = pd.concat([Omnia_dinten, Omnia_dinten_pg])


In [4]:
## PRODP ELEC & HEAT
Omnia_prodp_elecht_trav = Omnia_trav.loc[(Omnia_trav["Sector"] == "PWR") 
                          & (Omnia_trav["Enerp NeW"].isin(lst_cmd_todrop_dinten_pg))].copy()

Omnia_prodp_elecht_trav = Omnia_prodp_elecht_trav.groupby(by=["Region New", "Enerp NeW", "Year", "Unit"], as_index= False)["Value"].sum()
Omnia_prodp_elecht_trav["Variable_NeW"] = "PRODP"
Omnia_prodp_elecht_trav.loc[(Omnia_prodp_elecht_trav["Enerp NeW"] == "ELEC"), "Enerp NeW"] = "PG"
Omnia_prodp_elecht_trav.rename(columns={"Region New": "Region_NeW", "Enerp NeW": "Sector"}, inplace = True)
Omnia_prodp_elecht = Omnia_prodp_elecht_trav[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]


In [5]:
## PRODP except ELEC & HEAT
Omnia_prodp_trav = Omnia_trav.loc[(Omnia_trav["Sector"] == "UPS-MIN") & (Omnia_trav["Sub-sector"] == "Primary production") ].copy()
Omnia_prodp_trav = Omnia_prodp_trav.groupby(by=["Region New", "Enerp NeW", "Year", "Unit"], as_index = False)["Value"].sum()
Omnia_prodp_trav.rename(columns={"Region New": "Region_NeW"}, inplace=True)
Omnia_prodp_trav["Variable_NeW"] = "PRODP" + Omnia_prodp_trav["Enerp NeW"]
Omnia_prodp_trav["Sector"] = "Production"
Omnia_prodp = Omnia_prodp_trav[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]


In [6]:
## Other variables
## DNONENRP
Omnia_dnonenp_trav = Omnia_trav.loc[(Omnia_trav["Sector"] == "Non energy")].copy()
Omnia_dnonenp_trav =Omnia_dnonenp_trav.groupby(by=["Region New", "Sector", "Enerp NeW", "Unit", "Year"], as_index=False)["Value"].sum()
Omnia_dnonenp_trav.rename(columns={"Region New": "Region_NeW"}, inplace=True)
Omnia_dnonenp_trav["Variable_NeW"] = "DNONENP" + Omnia_dnonenp_trav["Enerp NeW"]
Omnia_dnonenp = Omnia_dnonenp_trav[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]


## IMPP/EXPP
Omnia_trade_trav = Omnia_trav.loc[(Omnia_trav["Sector"] == "TRADE")].copy()
Omnia_trade_trav = Omnia_trade_trav.groupby(by=["Region New", "Sub-sector", "Enerp NeW", "Unit", "Year"], as_index=False)["Value"].sum()
Omnia_trade_trav .loc[(Omnia_trade_trav["Sub-sector"] == "Export"), "Variable_NeW"] = "EXPP" + Omnia_trade_trav["Enerp NeW"]
Omnia_trade_trav .loc[(Omnia_trade_trav["Sub-sector"] == "Import"), "Variable_NeW"] = "IMPP" + Omnia_trade_trav["Enerp NeW"]
Omnia_trade_trav.drop(columns=["Sub-sector"])
Omnia_trade_trav.rename(columns={"Region New": "Region_NeW"}, inplace=True)
Omnia_trade_trav["Sector"] = Omnia_trade_trav["Enerp NeW"]
Omnia_trade = Omnia_trade_trav[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]


## DSTOCKP
lst_dstock = ["Consumption by stocks", "Production by stocks"]
Omnia_dstockp_trav = Omnia_trav.loc[(Omnia_trav["Sector"] == "UPS-MIN") & (Omnia_trav["Sub-sector"].isin(lst_dstock))].copy()
Omnia_dstockp_trav= Omnia_dstockp_trav.groupby(["Region New", "Enerp NeW", "Unit", "Year"], as_index=False)["Value"].sum()
Omnia_dstockp_trav["Variable_NeW"] = "DSTOCKP" + Omnia_dstockp_trav["Enerp NeW"]
Omnia_dstockp_trav["Sector"] = Omnia_dstockp_trav["Enerp NeW"]
Omnia_dstockp_trav.rename(columns={"Region New": "Region_NeW"}, inplace=True)
Omnia_dstockp = Omnia_dstockp_trav[["Variable_NeW", "Region_NeW", "Sector", "Year", "Value", "Unit"]]


In [7]:
# Merge all in one clean dataset
Omnia_eb = pd.concat([Omnia_dfinen, Omnia_dinten, Omnia_prodp_elecht, Omnia_prodp, Omnia_dnonenp, Omnia_trade, Omnia_dstockp])
Omnia_eb["Variable_NeW"].unique()
Omnia_eb.loc[(Omnia_eb["Variable_NeW"] == "PRODP") & (Omnia_eb["Sector"] == "PG"), "Variable_NeW"] = "PRODPELEC"
Omnia_eb.loc[(Omnia_eb["Variable_NeW"] == "PRODP") & (Omnia_eb["Sector"] == "HEAT"), "Variable_NeW"] = "PRODPHEAT"
Omnia_eb.reset_index(drop=True)

,Variable_NeW,Region_NeW,Sector,Year,Value,Unit
0,DFINENPBG,AU+WA,AGR,2019,0.0,TJ
1,DFINENPBG,BR,AGR,2019,0.0,TJ
2,DFINENPBG,CA,AGR,2019,7.0,TJ
3,DFINENPBG,CN,AGR,2019,0.0,TJ
4,DFINENPBG,DE+OE,AGR,2019,22156.0,TJ
...,...,...,...,...,...,...
4395,DSTOCKPGAS,WM,GAS,2019,2475.0,TJ
4396,DSTOCKPIW,WM,IW,2019,0.0,TJ
4397,DSTOCKPLBF,WM,LBF,2019,0.0,TJ
4398,DSTOCKPOIL,WM,OIL,2019,429929.0,TJ


### Split OMNIA regions to match with NeW regions

In [8]:
# Load dataframe to split OMNIA regions to map with NeW regions
nm_file = "Shares_tosplit_Omnia_Region.csv" # Share caulculated with raw UNDATA (see EnergyBalalnce_UNDATA file)
path_data = path_enr_data + nm_file
df_reg_tosplit = pd.read_csv(path_data, sep=";", header = 0, dtype= str)
df_reg_tosplit.rename(columns={"Region_splitted": "Region_NeW","Sector_OMNIA": "Sector"}, inplace=True)
df_reg_tosplit["Share"] = df_reg_tosplit["Share"].astype(float) # convert into a float

In [9]:
## Split OMNIA EB dataframe 
# List regions to split 
dict_tosplit = {"AU+WA": ["AU","WA1"], "DE+OE": ["DE", "OE1"], "FR+IT+ES+OE": ["FR","IT","ES","OE2"],
                 "ID+WA": ["ID", "WA2"], "UK+WE": ["UK","WE"]}
# Dataframe to store 
df_reg_splitted = pd.DataFrame()

# For each region to split, calculate the value for the region and the remianing, caution for "FR+IT+ES+OE", data storage in the dataframe a bit different
for agg, lst_co in dict_tosplit.items():
    df_agg = Omnia_eb.loc[Omnia_eb["Region_NeW"] == f"{agg}"].copy() # sub-set with region to split
    reg_rst = lst_co[-1] # last element of the list
    for reg in lst_co[:-1]:
        df_split = df_reg_tosplit.loc[(df_reg_tosplit["Region_NeW"] == f"{reg}")].copy() # sub-set of share dataframe
        df_merged = pd.merge(df_agg[["Variable_NeW", "Sector", "Year", "Value", "Unit"]],
                     df_split[[ "Variable_NeW", "Sector", "Share"]],
                     on = ["Variable_NeW", "Sector"], how= "left") # Merge both dataframe on all row (left) from df_agg 
        sec_mean_sh = df_merged.groupby('Sector')['Share'].transform(lambda x: x.fillna(x.mean())) # replace nan value in column "Share" by mean value by "Sector" column
        df_merged['Share'] = df_merged['Share'].fillna(sec_mean_sh) 
        df_merged.loc[df_merged['Share'].isna() & (df_merged['Value'] == 0), 'Share'] = 0 # reach remaining nan values
        df_merged.loc[df_merged['Share'].isna(), "Share"] = 0 # reach remaining nan values (case where value exists only for the second region)
        if reg_rst == "OE2": # In case region "FR+IT+ES+OE", store only "Value_rest" for France
            df_merged["Value_splitted"] = df_merged["Share"]*df_merged["Value"]
            df_merged["Value_rest"] = 0
            df_merged["Region_splitted"] = f"{reg}"
            df_merged["Region_rest"] = f"{reg_rst}"
            df_reg_splitted = pd.concat([df_reg_splitted, df_merged])
        else:
            df_merged["Value_splitted"] = df_merged["Share"]*df_merged["Value"]
            df_merged["Value_rest"] = (1-df_merged["Share"])*df_merged["Value"]
            df_merged["Region_splitted"] = f"{reg}"
            df_merged["Region_rest"] = f"{reg_rst}"
            df_reg_splitted = pd.concat([df_reg_splitted, df_merged])
            
            

# to add value_rest and value_splitted in one column
# for value_splitted
df_split_tmp = df_reg_splitted[["Variable_NeW", "Region_splitted", "Sector", "Year", "Value_splitted", "Unit"]].copy()
df_split_tmp = df_split_tmp.rename(columns={"Region_splitted": "Region_NeW", "Value_splitted": "Value"})
# for value_rest
df_rest_tmp = df_reg_splitted[["Variable_NeW", "Region_rest", "Sector", "Year", "Value_rest", "Unit"]].copy()
df_rest_tmp = df_rest_tmp.rename(columns={"Region_rest": "Region_NeW","Value_rest": "Value"})
# concatenate both
df_eb_splitted = pd.concat([df_split_tmp, df_rest_tmp], ignore_index=True)
df_eb_splitted = df_eb_splitted.groupby(by=["Variable_NeW", "Region_NeW", "Sector", "Year", "Unit"], as_index=False)["Value"].sum()



#  To calculate remaining value for region "FR+IT+ES+OE"           
df_tmp = df_reg_splitted.loc[df_reg_splitted["Region_splitted"].isin(["FR","IT","ES"])].copy() # sub-set to calculate the sum of spliited values
df_tmp = df_tmp.groupby(by=["Variable_NeW", "Sector", "Year", "Unit"], as_index=False)["Value_splitted"].sum() # sum of splitted values
df_tmp_val = df_reg_splitted.loc[df_reg_splitted["Region_splitted"].isin(["FR","IT","ES"])].copy() # sub-set
df_tmp_val = df_tmp_val[["Variable_NeW", "Sector", "Year", "Value", "Unit"]].drop_duplicates() # keep only uinqie combination of the columns

df_mrg = pd.merge(df_tmp, df_tmp_val, on=["Variable_NeW", "Sector", "Year", "Unit"], how="inner") # merge both dataframes
df_mrg["Value_rest"] = df_mrg["Value"] - df_mrg["Value_splitted"] # calculate remaining values
df_mrg["Region_NeW"] = "OE2" 
df_mrg = df_mrg[["Variable_NeW", "Region_NeW", "Sector", "Year", "Unit", "Value_rest"]]
df_mrg.rename(columns={"Value_rest": "Value"}, inplace=True)

# Suppress existing row for "OE2" & merge calculated remaining values
df_eb_splitted = df_eb_splitted.loc[df_eb_splitted["Region_NeW"] != "OE2"]
df_eb_splitted = pd.concat([df_eb_splitted,df_mrg])

# Aggregate rest by NeW regions
for reg in ["OE1", "OE2", "WA1", "WA2", "WA3"]:
    df_eb_splitted.loc[(df_eb_splitted["Region_NeW"] == f"{reg}"), "Region_NeW"] = reg[:2]
df_eb_splitted = df_eb_splitted.groupby(by=["Variable_NeW", "Region_NeW", "Sector", "Year", "Unit"], as_index=False)["Value"].sum()  

Omnia_eb_NeW = Omnia_eb.loc[~Omnia_eb["Region_NeW"].isin(["AU+WA","DE+OE","FR+IT+ES+OE","ID+WA","UK+WE"])].copy()
Omnia_eb_NeW = pd.concat([Omnia_eb_NeW, df_eb_splitted], ignore_index=True)
Omnia_eb_NeW = Omnia_eb_NeW.groupby(by=["Variable_NeW", "Region_NeW", "Sector", "Year", "Unit"], as_index=False)["Value"].sum()  

print("Percentage error between sum of all Omnia and splitted values : ", (Omnia_eb_NeW["Value"].sum()-Omnia_eb["Value"].sum())/Omnia_eb["Value"].sum()*100, "%")

Percentage error between sum of all Omnia and splitted values :  0.0 %


### Final raw data processing before calculation of monetary values

In [10]:
# Add a new column 'Variable_suffix' with the suffix after the specified prefixes, and replace the values in 'Variable_NeW' with the matched prefix
prefixes = ["DFINENP", "DINTENP", "DNONENP", "DSTOCKP", "EXPP", "IMPP", "PRODP"]
def extract_prefix_and_suffix(var):
    for prefix in prefixes:
        if var.startswith(prefix):
            return prefix, var[len(prefix):]
    return var, ""

Omnia_eb_ToNeW =  Omnia_eb_NeW.copy()
Omnia_eb_ToNeW[["Variable_NeW", "Enrp"]] = Omnia_eb_ToNeW["Variable_NeW"].apply(lambda x: pd.Series(extract_prefix_and_suffix(x)))
Omnia_eb_ToNeW[["Variable_NeW", "Enrp"]].drop_duplicates()
Omnia_eb_ToNeW[["Variable_NeW", "Region_NeW", "Sector", "Enrp", "Year", "Value", "Unit"]]

Omnia_eb_ToNeW.loc[Omnia_eb_ToNeW["Value"] == "PRODPHEAT", "Enrp"] = "HEAT"

# ⚠️ CAUTION CONVERT VALUES TO PJ
Omnia_eb_ToNeW["Value"] = Omnia_eb_NeW.copy()["Value"]*10**(-3) # Convert values to PJ # Use Omnia_eb_NeW to allow rerunning
Omnia_eb_ToNeW["Unit"] = "PJ" # Set unit to PJ

# ⚠️ CAUTION CONVERT NEGATIVE VALUES TO POSITIVE
Omnia_eb_ToNeW["Value"] = Omnia_eb_ToNeW["Value"].abs() # Convert negative values to positive

### From Omnia splitted Energy Balance to NeW

In [ ]:
# # ⚠️ TO ACTIVATE WHEN NOTEBOOK RUN STAND ALONE

# import pymrio
# import os

# # Sum demands to energy supply sectors for each region 
# # Load previously extracted and saved 
# save_folder_full = "E:/Work/DIAMOND/NeW/Data_Raw/EXIOBASE/pymrio/"
# exio19 = pymrio.load_all(path=save_folder_full)

# # Initial list and values
# BASEYEAR = "2019Y1"
# LOADYEAR = "2019Y1"
# SMPSTRY = "2015Y1"
# SMPENDY = "2050Y1"
# SMP = SMPSTRY + ":" + SMPENDY
# print(SMP)
# io.variables.sample = f'{SMP}'

# # Caution does not work when running the code several times, mrio base must be reloaded
# # Modify codes lists

# ## Rename final demands indexes
# lst_ini_FD = exio19.get_Y_categories()
# lst_upd_FD = ["CONSHV","CONSNPV","CONSGV","DINVV","DSTOCKV","DVALUEV","EXPTOTV"]
# dict_FD = dict(zip(lst_ini_FD, lst_upd_FD))
# exio19.rename_Y_categories(dict_FD)

# ## Rename sectors only with numbers (3-digit)
# # Do a list of numbers from 1 to 164 with zero(s) before
# lst_upd_Sect = [str(i).zfill(3) for i in range(1, 164)]
# lst_ini_Sect = exio19.get_sectors()
# dict_Sect = dict(zip(lst_ini_Sect, lst_upd_Sect))
# exio19.rename_sectors(dict_Sect)

# ## Mapping of the Sectors
# map_sect = pd.read_excel(os.path.join(save_folder_full,"Mapping_Sectors.xlsx"), dtype=str) # Excel file containing sectors' mapping for aggregation
# map_ini_sect = map_sect["Sect_cd_ini"]          # Initial sectors
# map_upd_sect = map_sect["Sect_cd_final"]        # Aggregated sectors
# dict_map_sect = dict(zip(map_ini_sect, map_upd_sect)) 
# exio19.rename_sectors(dict_map_sect)            # Replace names
# exio19.aggregate_duplicates()                   # Groupby

# ## Mapping of the regions
# map_reg = pd.read_excel(os.path.join(save_folder_full,"Mapping_Regions.xlsx"), dtype=str) # Excel file containing regions' mapping for aggregation
# map_ini_reg = map_reg["Reg_cd_ini"]             # Initial regions
# map_upd_reg = map_reg["Reg_cd_final"]           # Aggregated regions
# dict_map_reg = dict(zip(map_ini_reg, map_upd_reg))
# exio19.rename_regions(dict_map_reg)             # Replace names
# exio19.aggregate_duplicates()                   # Groupby

# lst_enr_sect = ["05","06","07","13","14","35","36","37","38","39"]
# DENRTOTV = exio19.Z.loc[(exio19.Z.index.get_level_values('sector').isin(lst_enr_sect))].groupby("sector").sum().sum()

In [ ]:
import re 

# # ⚠️ line to activate to when the notebook runs stand alone (to get consumption data)
# # Load consumption data
# path_cons = r"E:\Work\DIAMOND\NeW\Data_Raw\EXIOBASE"
# file_nm = "df_var_2019Y1.csv"
# df_cons_ener = pd.read_csv(os.path.join(path_cons, file_nm), sep=";")

# Clean and transform the dataframe
df_cons_ener[["variable", "region", "consp"]] = df_cons_ener["var"].str.split("_", expand=True) # Split the "var" column into three new columns, using "_" as the separator
df_cons_ener["sector"] = np.where(df_cons_ener["consp"] == "HTCL", "HC", "TR") # Column "sector" is defined as "TR" by defaut and "HT" for "HTCL"
df_cons_ener.drop(columns=["var","consp", "variable"], inplace=True) 
df_cons_ener = df_cons_ener.groupby(["region", "sector"])["value"].sum().reset_index() # Group by "region" and "sector" and sum the "value" column
df_cons_ener = df_cons_ener[["region", "sector", "value"]]
df_cons_ener = df_cons_ener.rename(columns={"region": "Region_NeW", "sector": "Sector_NeW", "value": "DENRTOTV"}) 

In [ ]:
# Collect, compile and process dataframes from EXIOBASE

# Row indexes into columns and rename colmuns for merging
DENRTOTV_reset = DENRTOTV.reset_index()  
DENRTOTV_reset = DENRTOTV_reset.rename(columns={"region": "Region_NeW", "sector": "Sector_NeW", 0: "DENRTOTV"})

# Merge DENRTOTV with df_cons_ener
DENRTOTV_tosplit = pd.concat([DENRTOTV_reset, df_cons_ener], ignore_index=True) 

### IDEM for production, exporrts and imports
df_exio_prod = exio19.x
df_exio_trade = exio19.get_gross_trade().totals
df_eco_tosplit = pd.concat([df_exio_trade,df_exio_prod], axis=1)
df_eco_tosplit = df_eco_tosplit.reset_index()
df_eco_tosplit = df_eco_tosplit.rename(columns={"region": "Region_NeW", "sector": "Sector_NeW"}) 



In [12]:
### DFINENP - Split OMINIA sectors with DENRTOTV
## Apply split of OMINIA sector "OTH OwnUses" to NeW sectors using previsously calculated share with raw UNData
# Load file previously calculated with raw UNData and transfrom dataframe
file_nm = "Shares_DFINENP_Other_Sectors.csv"
pathdata = path_enr_data + file_nm
df_oth_splited = pd.read_csv(pathdata , sep=";", header=0, dtype=str)
df_oth_splited["Value_TJ"] = df_oth_splited["Value_TJ"].astype(float) # convert into a float
df_oth_splited["Share"] = df_oth_splited["Share"].astype(float) # convert into a float
df_oth_splited.rename(columns={"Region_NeW_cd": "Region_NeW", "Variable_NeW_cd": "Variable_NeW", "Commodity_NeW_nm": "Enrp", "Sector": "Sector_NeW"}, inplace=True)
df_oth_splited[["Variable_NeW", "Region_NeW", "Sector_NeW", "Enrp", "Value_TJ", "Share"]]
df_oth_splited["Sector"] = "OTH OwnUses"

# Filter Omnia_eb_ToNeW for DFINENP and OTH OwnUses sectors
df_dfinenp_oth = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DFINENP") & (Omnia_eb_ToNeW["Sector"] == "OTH OwnUses")].copy()
df_dfinenp_oth = df_dfinenp_oth.merge(df_oth_splited[["Variable_NeW", "Region_NeW", "Sector", "Sector_NeW", "Enrp", "Share"]], 
                                       on=["Variable_NeW", "Region_NeW", "Sector", "Enrp"], how="left")

# Replace values with NaN by "35" for Sector_NeW and zero for "Share" (Only one case for non-zero in "Value":  (DFINENP, WE, OTH OwnUses, 2019, PJ, -20000.0, SBM, NaN, NaN)
# To verify: print(df_dfinenp_oth.loc[(df_dfinenp_oth["Value"] < -0.0001) & (df_dfinenp_oth["Sector_NeW"].isna())]) 
df_dfinenp_oth.loc[(df_dfinenp_oth["Sector_NeW"].isna()), "Sector_NeW"] = "35"
df_dfinenp_oth.loc[(df_dfinenp_oth["Share"].isna()), "Share"] = 0

# Calculate the corrected values for DFINENP OTH OwnUses as (Value*Share) and prepare the dataframe for merging
df_dfinenp_oth["Value_corr"] = df_dfinenp_oth["Value"]*df_dfinenp_oth["Share"]
df_dfinenp_oth.drop(columns=["Value", "Share"], inplace= True)
df_dfinenp_oth = df_dfinenp_oth.rename(columns={"Value_corr": "Value"})

## Create a new dataframe for DFINENP to split OMNIA sectors into NeW sectors
# Filter Omnia_eb_ToNeW for DFINENP and all except other sectors
df_dfinenp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DFINENP") & (~(Omnia_eb_ToNeW["Sector"] == "OTH OwnUses"))].copy()

# Dictionary to map OMNIA sectors to NeW sectors
dict_dfinenp = {"AGR": ["01","02","03","04"],
               "ICH": ["15","16","17"],
               "ICM": ["41"],
               "IIS": ["22"],
               "INF": ["23","24"],
               "INM": ["19","20","21"],
                "IOI": ["08","09","10","11","12","18","25","26","27","28","29","30","31","32","33","34","40"],
                "OTH OwnUses": ["05","06","07","08","13","14","17","38"], # Not use here
                "PWR OwnUses": ["35","36","37","39"],
                "OTH_TR": ["49"],
                "RAI": ["44"],
                # Caution may be in two different sectors, such as 
                "OTH": ["42","43","50","51","52","53","54","55","56","57","58","59","HC"],
                "RES": ["HC"],
                "ROA": ["45","TR"], 
                "SRV": ["42","43","50","51","52","53","54","55","56","57","58","59"],
                "NAD": ["46", "47"],
                "AVD": ["48"], 
                "Pipeline": ["45"],
                "NAB": ["46"],
                "AVB": ["48"]}


# Turn the dictionnay into a DataFrame for joining
df_map = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_dfinenp.items() for v in values])

# Merge with the original df_dfinenp
df_dfinenp = df_dfinenp.merge(df_map, on='Sector', how='left')

# Merge into df_dfinenp
df_dfinenp = df_dfinenp.merge(DENRTOTV_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")

df_dfinenp = df_dfinenp[["Variable_NeW", "Region_NeW", "Sector", "Sector_NeW", "Enrp", "Year", "Value", "Unit", "DENRTOTV"]]

# DFINENP values splitted by NeW regions and sectors
# Columns for grouping
group_cols = ["Variable_NeW", "Region_NeW", "Sector", "Enrp", "Year", "Unit"]
df = df_dfinenp  # alias for simplicity

# Sum by group - returning a serie with the same length than df)
df["DENRTOTV_sum"] = df.groupby(group_cols)["DENRTOTV"].transform("sum")

# # Weighted average
df["value_weighted"] = df["Value"]*(df["DENRTOTV"] / df["DENRTOTV_sum"])
df.loc[df["DENRTOTV_sum"] == 0, "value_weighted"] = np.nan

df_dfinenp = df


# Remove useless column   
df_dfinenp.drop(columns=["Sector", "DENRTOTV", "DENRTOTV_sum", "Value"], inplace=True)
df_dfinenp = df_dfinenp.rename(columns={"value_weighted": "Value"})

# Merge df_dfinenp with df_dfinenp_oth to include the OTH OwnUses sector
df_dfinenp_oth.drop(columns=["Sector"], inplace=True) 
df_dfinenp = pd.concat([df_dfinenp, df_dfinenp_oth])

# Verification
print("Percentage difference between sum of all Omnia values for DFINENP and splitted values for NeW sectors: ", 
      (((Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DFINENP"),"Value"].sum()*10**(-9)))
      -(df_dfinenp["Value"].sum()*10**(-9)))
     /(Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DFINENP"),"Value"].sum()*10**(-9))
        ,"%")

NameError: name 'DENRTOTV_tosplit' is not defined

In [ ]:
### DINTENP - Split OMINIA sectors with DENRTOTV
## Apply split of OMINIA sector "Other" to NeW sectors using previsously calculated share with raw UNData
# Load file previously calculated with raw UNData and transfrom dataframe
file_nm = "Shares_DINTENP_Other_Sectors.csv"
pathdata = path_enr_data + file_nm
df_oth_splited = pd.read_csv(pathdata, sep=";", header=0, dtype=str)
df_oth_splited["Value_TJ"] = df_oth_splited["Value_TJ"].astype(float) # convert into a float
df_oth_splited["Share"] = df_oth_splited["Share"].astype(float) # convert into a float
df_oth_splited.rename(columns={"Region_NeW_cd": "Region_NeW", "Variable_NeW_cd": "Variable_NeW", "Commodity_NeW_nm": "Enrp", "Sector": "Sector_NeW"}, inplace=True)
df_oth_splited[["Variable_NeW", "Region_NeW", "Sector_NeW", "Enrp", "Value_TJ", "Share"]]
df_oth_splited["Sector"] = "Other"


# Filter Omnia_eb_ToNeW for DINTENP and "Other" sectors
df_dintenp_oth = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DINTENP") & (Omnia_eb_ToNeW["Sector"] == "Other")].copy()
df_dintenp_oth = df_dintenp_oth.merge(df_oth_splited[["Variable_NeW", "Region_NeW", "Sector", "Sector_NeW", "Enrp", "Share"]], 
                                        on=["Variable_NeW", "Region_NeW", "Sector", "Enrp"], how="left")

# Replace values with NaN by "35" for Sector_NeW and zero for "Share" (no one case for non-zero in "Value")
# To verify: print(df_dintenp_oth.loc[(df_dintenp_oth["Value"] < -0.0001) & (df_dintenp_oth["Sector_NeW"].isna())]) 
df_dintenp_oth.loc[(df_dintenp_oth["Sector_NeW"].isna()), "Sector_NeW"] = "35"
df_dintenp_oth.loc[(df_dintenp_oth["Share"].isna()), "Share"] = 0

# Calculate the corrected values for DFINENP OTH OwnUses as (Value*Share) and prepare the dataframe for merging
df_dintenp_oth["Value_corr"] = df_dintenp_oth["Value"]*df_dintenp_oth["Share"]
df_dintenp_oth.drop(columns=["Value", "Share"], inplace= True)
df_dintenp_oth = df_dintenp_oth.rename(columns={"Value_corr": "Value"})

## Create a new dataframe for DFINENP to split OMNIA sectors into NeW sectors
# Filter Omnia_eb_ToNeW for DFINENP and all except other sectors
df_dintenp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DINTENP") & (~(Omnia_eb_ToNeW["Sector"] == "Other"))].copy()

# Dictionary to map OMNIA sectors to NeW sectors
dict_dintenp = {"HEAT": ["39"],
                "PG": ["35","36","37"],
                "Blast furnace": ["22"],
                "Coke oven": ["13"],
                "Refineries": ["14"],
                # Double
                "Liquefation": ["38"],
                "Other": ["05","06","07","13","14","17","38"]} # Not use here
                

# Turn the dictionnay into a DataFrame for joining
df_map = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_dintenp.items() for v in values])

# Merge with the original df_dintenp
df_dintenp = df_dintenp.merge(df_map, on='Sector', how='left')


# Merge into df_dintenp
df_dintenp = df_dintenp.merge(DENRTOTV_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")

df_dintenp = df_dintenp[["Variable_NeW", "Region_NeW", "Sector", "Sector_NeW", "Enrp", "Year", "Value", "Unit", "DENRTOTV"]]

# DINENTP values splitted by NeW regions and sectors
# Columns for grouping
group_cols = ["Variable_NeW", "Region_NeW", "Sector", "Enrp", "Year", "Unit"]
df = df_dintenp  # alias for simplicity      


# Sum by group - returning a serie with the same length than df)
df["DENRTOTV_sum"] = df.groupby(group_cols)["DENRTOTV"].transform("sum")

# Weighted average
df["value_weighted"] = df["Value"]*(df["DENRTOTV"] / df["DENRTOTV_sum"])
df.loc[df["DENRTOTV_sum"] == 0, "value_weighted"] = np.nan
# Case when DENRTOTV is zero and value_weighted is NaN, set value_weighted to Value
df.loc[(df["value_weighted"].isna()) & (df["DENRTOTV"] == 0), "value_weighted"] = df.loc[(df["value_weighted"].isna()) & (df["DENRTOTV"] == 0), "Value"]

df_dintenp = df

# Remove useless column   
df_dintenp.drop(columns=["Sector", "DENRTOTV", "DENRTOTV_sum", "Value"], inplace=True)
df_dintenp = df_dintenp.rename(columns={"value_weighted": "Value"})

# Merge df_dintenp with df_dintenp_oth to include the OTH OwnUses sector 
df_dintenp_oth.drop(columns=["Sector"], inplace=True) 
df_dintenp = pd.concat([df_dintenp, df_dintenp_oth])

# Verification
print("Percentage difference between sum of all Omnia values for DINTENP and splitted values for NeW sectors: ", 
      (((Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DINTENP"),"Value"].sum()*10**(-9)))
      -(df_dintenp["Value"].sum()*10**(-9)))
     /(Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DINTENP"),"Value"].sum()*10**(-9))
        ,"%")

In [ ]:
### DNONENP - Split OMINIA sectors with DENRTOTV
# Simplest case, only one OMNIA sector to split
# Filter Omnia_eb_ToNeW for DNONENP and "Non energy" sectors
df_dnonenp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DNONENP") ].copy()

# Dictionary to map OMNIA sectors to NeW sectors
dict_dnonenp = {"Non energy": ["15","17","41"]}

# Turn the dictionnay into a DataFrame for joining
df_map = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_dnonenp.items() for v in values])

# Merge with the original df_dintenp
df_dnonenp = df_dnonenp.merge(df_map, on='Sector', how='left')

# Merge into df_dnonenp
df_dnonenp = df_dnonenp.merge(DENRTOTV_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")
df_dnonenp = df_dnonenp[["Variable_NeW", "Region_NeW", "Sector", "Sector_NeW", "Enrp", "Year", "Value", "Unit", "DENRTOTV"]]

# DINENTP values splitted by NeW regions and sectors
# Columns for grouping
group_cols = ["Variable_NeW", "Region_NeW", "Sector", "Enrp", "Year", "Unit"]
df = df_dnonenp  # alias for simplicity      

# Sum by group - returning a serie with the same length than df)
df["DENRTOTV_sum"] = df.groupby(group_cols)["DENRTOTV"].transform("sum")

# Weighted average
df["value_weighted"] = df["Value"]*(df["DENRTOTV"] / df["DENRTOTV_sum"])
df.loc[df["DENRTOTV_sum"] == 0, "value_weighted"] = np.nan
# # # Case when DENRTOTV is zero and value_weighted is NaN, set value_weighted to Value
# # df.loc[(df["value_weighted"].isna()) & (df["DENRTOTV"] == 0), "value_weighted"] = df.loc[(df["value_weighted"].isna()) & (df["DENRTOTV"] == 0), "Value"]

df_dnonenp = df

# Remove useless column   
df_dnonenp.drop(columns=["Sector", "DENRTOTV", "DENRTOTV_sum","Value"], inplace=True)
df_dnonenp = df_dnonenp.rename(columns={"value_weighted": "Value"})

# Verification
print("Percentage difference between sum of all Omnia values for DINTENP and splitted values for NeW sectors: ", 
      (((Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DNONENP"),"Value"].sum()*10**(-9)))
      -(df_dnonenp["Value"].sum()*10**(-9)))
     /(Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DNONENP"),"Value"].sum()*10**(-9))
        ,"%")

In [ ]:
### PRODP/IMPP/EXPP/DSTOCKP - Split OMINIA sectors with DENRTOTV
# Filter Omnia_eb_ToNeW for DNONENP and "Non energy" sectors
df_impp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "IMPP") ].copy()
df_expp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "EXPP") ].copy()
df_dstockp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "DSTOCKP") ].copy()
df_prodp = Omnia_eb_ToNeW.loc[(Omnia_eb_ToNeW["Variable_NeW"] == "PRODP") ].copy()

# Dictionary to map OMNIA sectors to NeW sectors
dict_impp = {'COMB': ["13"], 'ELEC': ["37"], 'GAS': ["38"], 'LBF': ["14"], 'OIL': ["14"], 'SBM': ["01","03"]}
dict_expp = {'COMB': ["13"], 'ELEC': ["37"], 'GAS': ["38"], 'LBF': ["14"], 'OIL': ["14"], 'SBM': ["01","03"]}
dict_dstockp = {'COMB': ["13"], 'ELEC': ["35"], 'GAS': ["38"], 'LBF': ["14"], 'OIL': ["14"], 'SBM': ["01","03"], 'IW': ["34"]}
# Caution: for PRODP mapping on Enrp not Sector
dict_prodp =  {'BG': ["38"], 'COMB': ["13"], 'ELEC': ["35"], 'GAS': ["38"], 'GEO': ["39"], 
               'HEAT': ["39"], 'HYD': ["35"], 'IW': ["34"], 'LBF': ["14"], 'NUC': ["35"], 
               'OIL': ["14"], 'PV': ["35"], 'SBM': ["01","03"], 'SOLT':["39"], 'TID': ["35"], 'WD': ["35"]}

# Turn the dictionnay into a DataFrame for joining
df_map_impp = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_impp.items() for v in values])
df_map_expp = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_expp.items() for v in values])
df_map_dstockp = pd.DataFrame([{'Sector': k, 'Sector_NeW': v} for k, values in dict_dstockp.items() for v in values])
# Caution: for PRODP mapping on Enrp not Sector
df_map_prodp = pd.DataFrame([{'Enrp': k, 'Sector_NeW': v} for k, values in dict_prodp.items() for v in values])

# Merge with the original df_dintenp
df_impp = df_impp.merge(df_map_impp, on='Sector', how='left')
df_expp = df_expp.merge(df_map_expp, on='Sector', how='left')
df_dstockp = df_dstockp.merge(df_map_dstockp, on='Sector', how='left')
df_prodp = df_prodp.merge(df_map_prodp, on='Enrp', how='left')

# Merge df from EXIOBASE to split 
df_impp = df_impp.merge(df_eco_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")
df_expp = df_expp.merge(df_eco_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")
df_dstockp = df_dstockp.merge(df_eco_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")
df_prodp = df_prodp.merge(df_eco_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")

# Values splitted by NeW regions and sectors
# Columns for grouping
group_cols = ["Variable_NeW", "Region_NeW", "Sector", "Enrp", "Year", "Unit"]

# Sum by group - returning a serie with the same length than df)
df_impp["imports_sum"] = df_impp.groupby(group_cols)["imports"].transform("sum")
df_expp["exports_sum"] = df_expp.groupby(group_cols)["exports"].transform("sum")
df_dstockp["indout_sum"] = df_dstockp.groupby(group_cols)["indout"].transform("sum")
df_prodp["indout_sum"] = df_prodp.groupby(group_cols)["indout"].transform("sum")


# Weighted average
df_impp["value_weighted"] = df_impp["Value"]*(df_impp["imports"] / df_impp["imports_sum"])
df_expp["value_weighted"] = df_expp["Value"]*(df_expp["exports"] / df_expp["exports_sum"])
df_dstockp["value_weighted"] = df_dstockp["Value"]*(df_dstockp["indout"] / df_dstockp["indout_sum"])
df_prodp["value_weighted"] = df_prodp["Value"]*(df_prodp["indout"] / df_prodp["indout_sum"])
df_impp.loc[df_impp["imports_sum"] == 0, "value_weighted"] = np.nan
df_expp.loc[df_expp["exports_sum"] == 0, "value_weighted"] = np.nan
df_dstockp.loc[df_dstockp["indout_sum"] == 0, "value_weighted"] = np.nan
df_prodp.loc[df_prodp["indout_sum"] == 0, "value_weighted"] = np.nan

# Remove useless column   
df_impp.drop(columns=["Sector", "imports", "exports", "indout", "imports_sum", "Value"], inplace=True)
df_expp.drop(columns=["Sector", "imports", "exports", "indout", "exports_sum", "Value"], inplace=True)
df_dstockp.drop(columns=["Sector", "imports", "exports", "indout", "indout_sum", "Value"], inplace=True)
df_prodp.drop(columns=["Sector", "imports", "exports", "indout", "indout_sum", "Value"], inplace=True)


df_impp = df_impp.rename(columns={"value_weighted": "Value"})
df_expp = df_expp.rename(columns={"value_weighted": "Value"})
df_dstockp = df_dstockp.rename(columns={"value_weighted": "Value"})
df_prodp = df_prodp.rename(columns={"value_weighted": "Value"})

In [ ]:
#### MERGE ALL SPLIT ENERGY DATA
df_enr_physical = pd.concat([df_dfinenp, df_dintenp, df_dnonenp, df_impp, df_expp, df_dstockp, df_prodp], ignore_index=True)

In [ ]:
# Import energy prices and taxes
import openpyxl

pathpener = pathwork + "data_raw/energy/energy_prices/"
file_nm = "EnergyPrices_Processed_final.xlsx"
to_load = pathpener+file_nm
Enr_prices_raw = pd.read_excel(to_load, sheet_name="PENRHT_Final")
Enr_tax_raw = pd.read_excel(to_load, sheet_name="Tax_Final")

lst_sec_dfinen = [f"{i:02}" for i in range(1, 60)] + ["HC","TR"]

Enr_prices_long = Enr_prices_raw.melt(id_vars=[col for col in Enr_prices_raw.columns if col not in lst_sec_dfinen],
    value_vars= lst_sec_dfinen, var_name="sector", value_name="value_price")

Enr_tax_long = Enr_tax_raw.melt(id_vars=[col for col in Enr_tax_raw.columns if col not in lst_sec_dfinen],
    value_vars= lst_sec_dfinen, var_name="sector", value_name="value_tax")

Enr_prices_long

Enr_prices_agg = Enr_prices_long.merge(Enr_tax_long, on =["region","enrp", "sector"], how = "left")

Enr_prices_agg["value_tax"] = Enr_prices_agg["value_tax"].fillna(0)
Enr_prices_agg = Enr_prices_agg.rename(columns={"region": "Region_NeW", "sector": "Sector_NeW", "enrp": "Enrp", "value_price": "Value_price", "value_tax": "Value_tax"})



In [ ]:
# Merge energy data and energy prices and taxes
df_enr_trav = df_enr_physical.merge(Enr_prices_agg, on =["Region_NeW", "Sector_NeW", "Enrp"], how="left")
df_enr_trav["Value_enrp"] = df_enr_trav["Value"]*df_enr_trav["Value_price"]

# Grouping data to calculate the share of each DFINENP/DINTENP/DNONENP by region, sector and enrp
lst_dnonerp_to_split = ["DFINENP", "DINTENP", "DNONENP"]
df_trav = df_enr_trav.loc[df_enr_trav["Variable_NeW"].isin(lst_dnonerp_to_split)].copy()

group_cols = ["Region_NeW", "Sector_NeW", "Year", "Unit"]
df_trav["Value_enrp_sum"] = df_trav.groupby(group_cols)["Value_enrp"].transform("sum")
df_trav["Share_Value_enrp"] = df_trav["Value_enrp"]/df_trav["Value_enrp_sum"]
df_trav["Share_Value_enrp"] = df_trav["Share_Value_enrp"].fillna(0)

# Add DENRTOTV to calculate value in monetary unit
df_trav_merged = df_trav.merge(DENRTOTV_tosplit, on=["Region_NeW", "Sector_NeW"], how="left")
df_trav_merged
df_trav_merged["Value_splitted"] = df_trav_merged["Share_Value_enrp"]*df_trav_merged["DENRTOTV"]

# Verifying that sum of "Value_splitted" equalises "DENRTOTV"
df_trav_verif = df_trav_merged.groupby(group_cols)["Value_splitted"].sum()
df_trav_verif = df_trav_verif.to_frame()
df_trav_verif
df_tmp = DENRTOTV_tosplit.set_index(["Region_NeW", "Sector_NeW"])
df_trav_verif = df_trav_verif.join(df_tmp["DENRTOTV"])
df_trav_verif
df_trav_verif["Verif"] = df_trav_verif["DENRTOTV"]-df_trav_verif["Value_splitted"]
print("Verification - Sum of 'Value_splitted' above 'DENRTOTV':", len(df_trav_verif.loc[df_trav_verif["Verif"] < -0.001]))
print("Verification - Sum of 'Value_splitted' below 'DENRTOTV':", len(df_trav_verif.loc[df_trav_verif["Verif"] > 0.001]))
# print(df_trav_verif.loc[df_trav_verif["Verif"] > 0.001])

### Correction of exceptions: cases where DFINENP/DINTENP/DNONENP are null but not DENRTOTV
df_trav_corr = df_trav_merged.copy()
group_cols = ["Region_NeW", "Sector_NeW"]
df_trav_corr["Value_splitted_verif"] = df_trav_corr.groupby(group_cols)["Value_splitted"].transform("sum")
df_trav_corr["Verif"] = df_trav_corr["DENRTOTV"]-df_trav_corr["Value_splitted_verif"]
## Sector 38 # Assuming 100% to Gas, and 0% to Comb
mask_38_gas = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == "38") & (df_trav_corr["Enrp"] == "GAS"))
mask_38_comb = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == "38") & (df_trav_corr["Enrp"] == "COMB"))
# Monetary units
df_trav_corr.loc[mask_38_gas, "Value_splitted"] = df_trav_corr.loc[mask_38_gas, "DENRTOTV"]
df_trav_corr.loc[mask_38_comb, "Value_splitted"] = 0
# Physcial units
mask_38_gas_us = ((df_trav_corr["Region_NeW"] == "US") & (df_trav_corr["Sector_NeW"] == "38") & (df_trav_corr["Enrp"] == "GAS"))
corr_factor = df_trav_corr.loc[mask_38_gas_us]["Value"].values/df_trav_corr.loc[mask_38_gas_us]["Value_splitted"].values
df_trav_corr.loc[mask_38_gas, "Value"] = df_trav_corr.loc[mask_38_gas, "Value_splitted"]*corr_factor
df_trav_corr.loc[mask_38_comb, "Value"] = 0

## Sector 49  # Assuming 85% Oil and 15% Elec 
mask_49_oil = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == "49") & (df_trav_corr["Enrp"] == "OIL"))
mask_49_elec = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == "49") & (df_trav_corr["Enrp"] == "ELEC"))
# Monetary units
df_trav_corr.loc[mask_49_oil, "Value_splitted"] = df_trav_corr.loc[mask_49_oil, "DENRTOTV"]*0.85
df_trav_corr.loc[mask_49_elec, "Value_splitted"] = df_trav_corr.loc[mask_49_elec, "DENRTOTV"]*0.15
# Physcial units
mask_49_oil_de = ((df_trav_corr["Region_NeW"] == "DE") & (df_trav_corr["Sector_NeW"] == "49") & (df_trav_corr["Enrp"] == "OIL"))
mask_49_elec_fr = ((df_trav_corr["Region_NeW"] == "FR") & (df_trav_corr["Sector_NeW"] == "49") & (df_trav_corr["Enrp"] == "ELEC"))
corr_factor_oil = df_trav_corr.loc[mask_49_oil_de]["Value"].values/df_trav_corr.loc[mask_49_oil_de]["Value_splitted"].values
corr_factor_elec = df_trav_corr.loc[mask_49_elec_fr]["Value"].values/df_trav_corr.loc[mask_49_elec_fr]["Value_splitted"].values
df_trav_corr.loc[mask_49_oil, "Value"] = df_trav_corr.loc[mask_49_oil, "Value_splitted"]*corr_factor_oil
df_trav_corr.loc[mask_49_elec, "Value"] = df_trav_corr.loc[mask_49_elec, "Value_splitted"]*corr_factor_elec

# For sectors 19/20/21/23 # Issue ony for ID, using WA for corrections
lst_sec_corr = ["19","20","21","23", "24"]
for sec in lst_sec_corr:
    mask_id = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == sec))
    for enr in df_trav_corr.loc[mask_id]["Enrp"].unique():
        mask_id_sec = ((df_trav_corr["Verif"] > 0.001) & (df_trav_corr["Sector_NeW"] == sec) & (df_trav_corr["Enrp"] == enr))
        mask_wa_sec = ((df_trav_corr["Region_NeW"] == "WA") & (df_trav_corr["Sector_NeW"] == sec) & (df_trav_corr["Enrp"] == enr))
        sh_val = df_trav_corr.loc[mask_wa_sec, "Share_Value_enrp"].values
        df_trav_corr.loc[mask_id_sec, "Value_splitted"] = df_trav_corr.loc[mask_id_sec, "DENRTOTV"]*sh_val
        if df_trav_corr.loc[mask_wa_sec, "Value_splitted"].values == 0:
            conv_factor = 0
        else:
            conv_factor = df_trav_corr.loc[mask_wa_sec, "Value"].values/df_trav_corr.loc[mask_wa_sec, "Value_splitted"].values
        df_trav_corr.loc[mask_id_sec, "Value"] = df_trav_corr.loc[mask_id_sec, "Value_splitted"]*conv_factor


# Second Verification
df_trav_verif = df_trav_corr.groupby(group_cols)["Value_splitted"].sum()
df_trav_verif = df_trav_verif.to_frame()
df_trav_verif
df_tmp = DENRTOTV_tosplit.set_index(["Region_NeW", "Sector_NeW"])
df_trav_verif = df_trav_verif.join(df_tmp["DENRTOTV"])
df_trav_verif
df_trav_verif["Verif"] = df_trav_verif["DENRTOTV"]-df_trav_verif["Value_splitted"]
print("Verification - Sum of 'Value_splitted' above 'DENRTOTV':", len(df_trav_verif.loc[df_trav_verif["Verif"] < -0.001]))
print("Verification - Sum of 'Value_splitted' below 'DENRTOTV':", len(df_trav_verif.loc[df_trav_verif["Verif"] > 0.001]))
print(df_trav_verif.loc[df_trav_verif["Verif"] > 0.001])

df_trav_merged = df_trav_corr.copy()

In [ ]:
# To clean and reorder the dataframe with variables in rows
df_ener_clean = df_trav_merged.copy()
df_ener_clean = df_ener_clean.drop(columns=["Value_enrp","Value_enrp_sum","Share_Value_enrp","DENRTOTV", "Value_splitted_verif", "Verif"])

# Melt avoiding columns name conflict
df_melted = df_ener_clean.melt(id_vars=["Variable_NeW", "Region_NeW", "Sector_NeW", "Enrp", "Year", "Unit"],
                               value_vars=["Value_price", "Value_tax", "Value_splitted"],
                            var_name="Type", value_name="Value_melted")

# Mapping vectorised of new "Variable_New"
df_melted["Variable_NeW"] = np.select([df_melted["Type"].eq("Value_price"),
                                       df_melted["Type"].eq("Value_tax"),
                                       df_melted["Type"].eq("Value_splitted") & df_melted["Variable_NeW"].eq("DFINENP"),
                                       df_melted["Type"].eq("Value_splitted") & df_melted["Variable_NeW"].eq("DINTENP"),
                                      df_melted["Type"].eq("Value_splitted") & df_melted["Variable_NeW"].eq("DNONENP"),],
                    ["PENRHT", "TAXENR", "DFINENV", "DINTENV", "DNONENV"], default=df_melted["Variable_NeW"])

# Finalisation: Remove "Type" and rename Value_melt -> Value
df_enr = (df_melted.drop(columns=["Type"]).rename(columns={"Value_melt": "Value"}).reset_index(drop=True))

df_enr.loc[df_enr["Variable_NeW"] == "PENRHT", "Unit"] = "M€/PJ"
df_enr.loc[df_enr["Variable_NeW"] == "TAXENR", "Unit"] = "M€/PJ"
df_enr.loc[df_enr["Variable_NeW"] == "DFINENV", "Unit"] = "M€"
df_enr.loc[df_enr["Variable_NeW"] == "DINTENV", "Unit"] = "M€"
df_enr.loc[df_enr["Variable_NeW"] == "DNONENV", "Unit"] = "M€"